# Run PDF parsing from Python

Set the variables below, then run the cells in order. Choose **Paddle** or **MinerU**; processing runs on Azure.

First follow the package installation steps in [README.md](README.md), select that Python environment as this notebook's kernel, and obtain `config.json` from your administrator. The Azure endpoint must already be deployed.

In [ ]:
from pathlib import Path

# Settings: edit these before running. Absolute paths also work.
PARSER = "paddle"                           # "paddle" or "mineru"
INPUT_FOLDER = Path("./pdfs")                # Select PDFs directly inside; ignore other files/subfolders
OUTPUT_FOLDER = Path(f"./{PARSER}-results")  # Local folder for all exports, logs and the report
CONFIG_FILE = Path("./config.json")          # Workspace configuration from your administrator

# Leave None for a NEW batch. To retrieve an existing batch, use its receipt:
# RESUME_RECEIPT = Path("./runs/paddle-...json")
RESUME_RECEIPT = None

WAIT_FOR_RESULTS = True  # New batches: False returns after submission; True waits and downloads
SIGN_IN = "browser"     # "browser", "device-code", or "az-login" (an existing az login session)

print("Relative paths start from:", Path.cwd())


## Submit or retrieve results

Run the cell below and sign in when prompted. With `RESUME_RECEIPT = None`, this submits a **new batch**: rerunning it can submit another batch once the endpoint is free. Only PDFs directly inside `INPUT_FOLDER` are selected; other files and subfolders are ignored. Use `Path(".")` for the current directory. The client uploads a temporary copy containing only the selected PDFs, then removes it. Allow enough local disk space for that copy and keep the selected PDFs unchanged until submission finishes.

With a receipt set, this retrieves the **existing batch** and never resubmits it. It waits if necessary, then downloads all available results. `INPUT_FOLDER`, `CONFIG_FILE` and `WAIT_FOR_RESULTS` are ignored in this mode; the receipt supplies the original workspace configuration. Keep `PARSER` set to the parser used for that batch.

**Wait for `Submitted` before closing your laptop.** Keep the printed receipt path. Once accepted, the Azure job continues even if you close the notebook or interrupt the cell. Batch errors and unconfirmed PDFs are printed below; inspect `report.json` and `parser.log` after download. A parser error may stop the batch; no PDFs are automatically retried.

In [ ]:
from pdf_parsers.client import main as run_parser

if PARSER not in ("paddle", "mineru"):
    raise ValueError('PARSER must be "paddle" or "mineru".')

sign_in_flags = {"browser": [], "device-code": ["--device-code"], "az-login": ["--az-login"]}
if SIGN_IN not in sign_in_flags:
    raise ValueError('SIGN_IN must be "browser", "device-code", or "az-login".')

if RESUME_RECEIPT is None:
    arguments = [str(INPUT_FOLDER), "--config", str(CONFIG_FILE)]
    if not WAIT_FOR_RESULTS:
        arguments.append("--no-wait")
else:
    arguments = ["--resume", str(RESUME_RECEIPT)]

arguments += ["--output", str(OUTPUT_FOLDER)] + sign_in_flags[SIGN_IN]
exit_code = run_parser(PARSER, arguments)
if exit_code != 0:
    raise RuntimeError("The client did not complete successfully. Read the messages above; check the receipt and Azure job before submitting again.")


## Download later

In the settings cell, replace `RESUME_RECEIPT = None` with `RESUME_RECEIPT = Path("the printed receipt path")`, choose `OUTPUT_FOLDER`, then rerun both code cells. This works after restarting the notebook and can download the files again while they remain in Azure storage.

For another new batch, set `RESUME_RECEIPT = None`, update the input/output folders and rerun both code cells. Use a separate output folder for each batch to keep its reports and exports together.